# v0.20.0 — Typed live queries

v0.19.0 delivered live queries as the server's raw envelopes. v0.20.0 adds the **typed** layer,
with the same API as the full SurrealDB-ORM: `QuerySet.live()` yields `ModelChangeEvent` objects
whose `instance` is a model instance, the queryset's **filters are applied by the server**, and
`live(diff=True)` streams patches instead of whole records.

It runs on **both SurrealDB 2.x and 3.x**. One server difference shapes the design: SurrealDB 2.x
does not carry bound parameters into a live query (the filter silently matches nothing), so for
live queries the ORM writes each filter value into the statement as a SurrealQL literal. Section 7
shows the statement it sends.

Everything below runs top to bottom on either server.

## 1. Connect

In [1]:
import asyncio
import contextlib
import os

from surreal_orm_lite import LiveAction, SurrealDBConnectionManager

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")

# Live queries require WebSocket.
SurrealDBConnectionManager.set_connection(
    url=f"ws://{HOST}:{PORT}/rpc",
    user="root",
    password="root",
    namespace="examples",
    database="examples",
)
print("Connection configured:", SurrealDBConnectionManager.is_connection_set())

Connection configured: True


## 2. A model, and an idempotent reset

`label` is stored in the `label_col` column (a v0.18.0 field alias) — filters and
`changed_fields` keep speaking Python names.

In [2]:
from pydantic import Field

from surreal_orm_lite import BaseSurrealModel


class Ticket(BaseSurrealModel):
    id: str
    status: str = ""
    priority: int = 0
    label: str = Field(default="", alias="label_col")


client = await SurrealDBConnectionManager.get_client()

with contextlib.suppress(Exception):
    await client.query("REMOVE TABLE Ticket;", {})
# SurrealDB 3.x refuses a live query on an undefined table, so define it up front.
await client.query("DEFINE TABLE Ticket SCHEMALESS;", {})
print("Table Ticket ready")

Table Ticket ready


## 3. `live()` — typed events, filtered by the server

The writes happen in a separate task, the way changes made by another client would arrive. Only
the records matching the filter are reported, and each event carries a `Ticket`. Every read is
wrapped in a timeout, so the notebook can never hang on a quiet stream.

In [3]:
async def writer() -> None:
    await asyncio.sleep(0.2)
    t1 = Ticket(id="t1", status="open", priority=1)
    await t1.save()
    await Ticket(id="t2", status="closed", priority=5).save()  # outside the filter: never reported
    t3 = Ticket(id="t3", status="open", priority=7)
    await t3.save()
    await t1.merge(priority=2)
    await t3.delete()


async def watch_open_tickets() -> None:
    async with Ticket.objects().filter(status="open").live() as stream:
        task = asyncio.create_task(writer())
        seen = 0
        async for event in stream:
            if task.done() and task.exception():
                raise task.exception()
            ticket = event.instance  # a Ticket, not a dict
            print(f"{event.action:<7} {event.record_id:<10} priority={ticket.priority}")
            seen += 1
            if seen == 4:
                break
        await task


await asyncio.wait_for(watch_open_tickets(), timeout=15)

CREATE  Ticket:t1  priority=1
CREATE  Ticket:t3  priority=7
UPDATE  Ticket:t1  priority=2
DELETE  Ticket:t3  priority=7


`t2` never shows up: it does not match `status = 'open'`. Two more consequences of a
server-side filter are worth knowing — a record that *enters* the filter through an update
arrives as `UPDATE` (not `CREATE`), and one that *leaves* it reports nothing at all:

In [4]:
async def enter_and_leave() -> None:
    async with Ticket.objects().filter(status="open").live() as stream:
        await (await Ticket.objects().get("t2")).merge(status="open")  # t2 enters the filter
        await (await Ticket.objects().get("t1")).merge(status="closed")  # t1 leaves it: silent
        event = await asyncio.wait_for(anext(stream), timeout=5)
        print(event.action, event.record_id)
        try:
            await asyncio.wait_for(anext(stream), timeout=1)
        except TimeoutError:
            print("nothing for t1 leaving the filter")


await enter_and_leave()

UPDATE Ticket:t2


nothing for t1 leaving the filter


## 4. Diff mode — what changed

`live(diff=True)` asks the server for patches. An `UPDATE` then carries only the changes:
`changed_fields` lists them (Python names — note `label`, not `label_col`), `instance` holds just
the `id`, and `raw` is the patch list. A changed string arrives as SurrealDB's `change` op, whose
value is diff-match-patch text rather than the new string.

In [5]:
async def diff_demo() -> None:
    async with Ticket.objects().live(diff=True) as stream:
        d1 = Ticket(id="d1", status="open", priority=1, label="draft")
        await d1.save()
        await d1.merge(priority=3, label="final")
        created = await asyncio.wait_for(anext(stream), timeout=5)
        updated = await asyncio.wait_for(anext(stream), timeout=5)
    print("CREATE carries the whole record:", created.instance)
    print("UPDATE changed_fields:", sorted(updated.changed_fields))
    print("UPDATE instance:", updated.instance.id)
    for patch in updated.raw:
        print("  patch:", patch["op"], patch["path"])


await diff_demo()

CREATE carries the whole record: id='d1' status='open' priority=1 label='draft'
UPDATE changed_fields: ['label', 'priority']
UPDATE instance: d1
  patch: change /label_col
  patch: replace /priority


## 5. The `post_live_change` signal

Handlers connected to `post_live_change` run for every event a `live()` stream yields — so they
react to changes made by *any* client. They run in the background and never stall the stream.

In [6]:
from surreal_orm_lite import post_live_change

received = []


@post_live_change.connect(Ticket)
async def on_ticket_change(sender, instance, action, record_id, changed_fields, **kwargs):
    received.append((action, record_id))


async def signal_demo() -> None:
    async with Ticket.objects().filter(status="urgent").live() as stream:
        await Ticket(id="s1", status="urgent", priority=9).save()
        await asyncio.wait_for(anext(stream), timeout=5)
        await asyncio.sleep(0.1)  # let the background handler run


await signal_demo()
post_live_change.disconnect(on_ticket_change, Ticket)
print("handler saw:", received)

handler saw: [(<LiveAction.CREATE: 'CREATE'>, 'Ticket:s1')]


## 6. Raw envelopes with `watch()`

`watch()` takes the same filters and `diff=` but yields the server's envelope as a plain dict —
useful when you want the payload exactly as SurrealDB sent it.

In [7]:
async def raw_demo() -> None:
    async with Ticket.objects().filter(priority__gte=8).watch() as stream:
        await Ticket(id="r1", status="open", priority=8).save()
        notif = await asyncio.wait_for(anext(stream), timeout=5)
    print(sorted(notif))
    print(notif["action"], notif["result"]["priority"])


await raw_demo()

['action', 'id', 'record', 'result', 'session']
CREATE 8


## 7. What the ORM sends — and why values are inlined

On SurrealDB 2.x, `LIVE SELECT … WHERE status = $s` is accepted, but `$s` reads as `NONE` when a
change is evaluated, so the filter matches nothing and no error is raised. The ORM therefore
writes live-query filter values as SurrealQL literals, through a closed encoder that refuses any
type it has not verified. This is what a filtered `live()` compiles to:

In [8]:
from decimal import Decimal

from surreal_orm_lite import Q, Var

qs = (
    Ticket.objects()
    .filter(Q(status="open") | Q(label="it's urgent"), priority__gte=Var("floor"))
    .variables(floor=Decimal("2.5"))
)
print(qs._compile_live(False))

try:
    Ticket.objects().filter(status=object()).live()
except TypeError as exc:
    print("refused:", str(exc).split(".")[0])

LIVE SELECT * FROM Ticket WHERE priority >= 2.5dec AND (status = 'open' OR label_col = 'it\'s urgent');
refused: $_f0: Cannot inline a value of type object into a live query filter


## 8. Cleanup

In [9]:
with contextlib.suppress(Exception):
    await client.query("REMOVE TABLE Ticket;", {})
await SurrealDBConnectionManager.close_connection()
print("done")

done
